# Chapter 9: Preference Data and Reward Models

Companion notebook for *Practical AI Safety from First Principles*, Chapter 9.

Chapters 2-8 evaluated behaviour after a model already existed. This chapter moves one step earlier: we build the reward model that later chapters will use to shape behaviour in the first place. A pairwise preference ("response A was preferred to response B") is a relational target, not an absolute quality score, and that single change creates a much richer modelling problem than the binary safety labels from Chapter 2. We treat a reward model exactly the way Chapters 3-4 treated a classifier: start with the simplest version (a linear model on frozen representations, which turns out to be logistic regression in disguise), understand it completely, then add a small amount of complexity.

By the end we will have: a full audit of PKU-SafeRLHF's helpfulness/safety preference structure (agreement rate, safety-combination breakdown, severity, response-source effects, prompt-disjoint splitting); a frozen-representation reward model built two ways (scikit-learn logistic regression on difference features, and an equivalent PyTorch scalar head trained with the actual Bradley-Terry loss); a full reward-model audit (pairwise accuracy, margins, calibration, bootstrap intervals, safety-combination and severity slices, the helpfulness/safety conflict rows specifically); a Pareto-frontier view of the two objectives; and an external generalisation check on RewardBench 2.

**A note on runtime.** The expensive step here is encoding text with Qwen3-0.6B (roughly 0.4 seconds per response on Apple Silicon MPS), not training the reward head itself, once representations are cached, fitting and evaluating both reward heads takes seconds. `N_FIT` / `N_VAL` default to a few hundred pairs, enough to see every part of the pipeline work correctly, well short of the book's suggested 5,000-10,000; raise them if you have the time, encoding scales linearly.

## 9.1 Why preference data changes the problem

A pairwise label tells us which of two responses was preferred *in that specific comparison*, under a *specific* annotation rule, it is not a universal quality score for either response. A response can win a comparison against a weak alternative without being good in any absolute sense. PKU-SafeRLHF makes a second distinction explicit: helpfulness and harmlessness are graded as **two separate preference dimensions**, `better_response_id` and `safer_response_id`, and they do not have to agree. A response can be more helpful while being less safe, that is not a data quality problem, it is exactly the tension a multi-objective post-training system has to resolve, and collapsing the two into one number before measuring where they disagree throws away the most useful part of the dataset.

One more thing worth holding onto before we touch any code: with more than two candidates per prompt, pairwise preferences can form a cycle (A beats B, B beats C, C beats A), because different qualities dominate different comparisons. A single scalar reward model is then being asked to compress a genuinely non-scalar preference structure into one ordering. Some disagreement we will find later is the model being wrong. Some of it is the data being irreducibly non-scalar. Telling those apart is part of the audit, not an excuse to skip it.

## 9.2 Audit PKU-SafeRLHF before training anything

In [1]:
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
import pandas as pd

DATASET_NAME = "PKU-Alignment/PKU-SafeRLHF"
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 73907
    })
    test: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 8211
    })
})
['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_ca

In [2]:
train_df = dataset["train"].to_pandas().copy()
test_df = dataset["test"].to_pandas().copy()

def canonicalise_preferences(df):
    out = df.copy()
    out["help_pref"] = out["better_response_id"].astype(int)
    out["safe_pref"] = out["safer_response_id"].astype(int)
    out["help_safe_agree"] = (out["help_pref"] == out["safe_pref"])
    return out

train_df = canonicalise_preferences(train_df)
test_df = canonicalise_preferences(test_df)
print(train_df.shape, test_df.shape)

(73907, 19) (8211, 19)


`help_pref = 0` means response 0 was preferred for helpfulness. It is a response identifier, not a claim that response 0 is good in any absolute sense.

In [3]:
agreement = train_df["help_safe_agree"].value_counts(normalize=True).rename(index={True: "agree", False: "disagree"})
print(agreement)

help_safe_agree
agree       0.759184
disagree    0.240816
Name: proportion, dtype: float64


The disagreement rows are the interesting ones, they are exactly the examples where a single combined reward would have to resolve a real trade-off rather than reinforce the same winner twice.

In [4]:
safety_pairs = pd.crosstab(train_df["is_response_0_safe"], train_df["is_response_1_safe"], normalize="all")
print(safety_pairs)

is_response_1_safe     False     True 
is_response_0_safe                    
False               0.441853  0.084660
True                0.061645  0.411842


In [5]:
train_df["safety_pair"] = list(zip(train_df["is_response_0_safe"], train_df["is_response_1_safe"]))

agreement_by_pair = (
    train_df.groupby("safety_pair")["help_safe_agree"]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
)
print(agreement_by_pair)

                count      mean
safety_pair                    
(False, False)  32656  0.753215
(True, True)    30438  0.739865
(False, True)    6257  0.856800
(True, False)    4556  0.796971


If agreement is near-total when exactly one response is unsafe (the safety ordering is easy, harmless beats harmful) but drops when both responses share the same safety label, that tells us precisely where the multi-objective tension concentrates: not "safe vs. unsafe" in general, but choosing between two responses that are already on the same side of that line.

In [6]:
severity_cols = ["response_0_severity_level", "response_1_severity_level"]
print(train_df[severity_cols].describe())

       response_0_severity_level  response_1_severity_level
count               73907.000000               73907.000000
mean                    1.068410                   1.010283
std                     1.081891                   1.061667
min                     0.000000                   0.000000
25%                     0.000000                   0.000000
50%                     1.000000                   1.000000
75%                     2.000000                   2.000000
max                     3.000000                   3.000000


In [7]:
source_table = pd.crosstab(train_df["response_0_source"], train_df["response_1_source"])
print(source_table)

response_1_source  Alpaca-7B  Alpaca2-7B  Alpaca3-8B
response_0_source                                   
Alpaca-7B              27393           0           0
Alpaca2-7B                 0       25564           0
Alpaca3-8B                 0           0       20950


In [8]:
source_pref = (
    train_df.groupby(["response_0_source", "response_1_source"])
    .agg(
        n=("help_pref", "size"),
        response0_help_win_rate=("help_pref", lambda s: (s == 0).mean()),
        response0_safe_win_rate=("safe_pref", lambda s: (s == 0).mean()),
    )
    .sort_values("n", ascending=False)
)
print(source_pref.head(10))

                                         n  response0_help_win_rate  \
response_0_source response_1_source                                   
Alpaca-7B         Alpaca-7B          27393                 0.357756   
Alpaca2-7B        Alpaca2-7B         25564                 0.354913   
Alpaca3-8B        Alpaca3-8B         20950                 0.370501   

                                     response0_safe_win_rate  
response_0_source response_1_source                           
Alpaca-7B         Alpaca-7B                         0.287373  
Alpaca2-7B        Alpaca2-7B                        0.284893  
Alpaca3-8B        Alpaca3-8B                        0.300621  


A strong source effect is not automatically leakage, one generator may genuinely write better responses. The question, which we test properly in section 9.6, is whether the reward model still ranks correctly when the source pairing changes, or whether it partly learned to recognise which model wrote the text.

In [9]:
for col in ["response_0_sha256", "response_1_sha256"]:
    print(col, "duplicated:", train_df[col].duplicated().sum())
print("Repeated prompts in train:", train_df["prompt"].duplicated().sum())

response_0_sha256 duplicated: 2048
response_1_sha256 duplicated: 2061
Repeated prompts in train: 35266


### A prompt-disjoint validation split

In [10]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
fit_idx, val_idx = next(splitter.split(train_df, groups=train_df["prompt"]))

fit_df = train_df.iloc[fit_idx].reset_index(drop=True)
val_df = train_df.iloc[val_idx].reset_index(drop=True)

overlap = set(fit_df["prompt"]) & set(val_df["prompt"])
print("fit:", fit_df.shape, "val:", val_df.shape, "prompt overlap:", len(overlap))

fit: (62025, 20) val: (11882, 20) prompt overlap: 0


Splitting by prompt group rather than by row means the validation experiment genuinely asks whether the reward model ranks responses for prompts it never saw during fitting, a random row split would let the same prompt (with different candidate responses) appear on both sides, and the reported accuracy would be measuring something closer to memorisation than generalisation.

## 9.3 Bradley-Terry preference modelling from first principles

For a scalar reward `r(x, y)`, the Bradley-Terry model says the probability that the winner beats the loser depends only on the **margin** between their scores: `P(w beats l) = sigmoid(r_w - r_l)`. If the margin is zero, the model is indifferent. A strongly positive margin means confident agreement with the label; a negative margin means the ranking is wrong. The negative log-likelihood of the observed preference is exactly the logistic loss on that margin, `-log(sigmoid(r_w - r_l))`, which is why this is often called the pairwise or Bradley-Terry loss rather than anything more exotic.

In [11]:
import torch
import torch.nn.functional as F

def pairwise_reward_loss(reward_w, reward_l):
    margin = reward_w - reward_l
    return -F.logsigmoid(margin).mean()

# A worked sanity check: a correct, well-separated pair; a barely-separated pair; a wrongly-ordered pair.
reward_w = torch.tensor([3.0, 0.2, -0.8])
reward_l = torch.tensor([0.5, 0.1, 0.4])
per_pair_loss = -F.logsigmoid(reward_w - reward_l)
print(per_pair_loss)
print("mean:", float(per_pair_loss.mean()))

tensor([0.0789, 0.6444, 1.4633])
mean: 0.7288562655448914


Adding the same constant to both scores in a pair leaves the margin, and therefore the loss, completely unchanged. Pairwise training identifies score *differences*, not an absolute origin, a reward model centred at 0 and one centred at 100 can make identical predictions if their margins agree. That is the technical reason a raw reward value should never be read as a stand-alone quantity of "how much a human would like this", only the comparison between two scores under the same model carries meaning.

One more connection worth seeing explicitly: if the reward is linear in a fixed representation, `r(h) = w . h`, then for two responses to the same prompt the margin is `w . (h_w - h_l)`, the intercept cancels, and the Bradley-Terry probability becomes exactly the logistic-regression sigmoid applied to the *difference* between the two response representations. A pairwise reward model is logistic regression on a differenced feature vector. We exploit that directly in the next section.

## 9.4 Build the first reward model with familiar data-science tools

### Use a frozen language model as the feature extractor

In [12]:
# Sample sizes: encoding is the expensive step here (about 0.4s per response on Apple Silicon MPS),
# not fitting the reward head. These defaults keep the notebook's total runtime reasonable; raise
# them toward the book's suggested 5,000-10,000 pairs if you have the time.
N_FIT = 500
N_VAL = 150


In [13]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, dtype="auto", device_map="auto")
encoder.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("device:", encoder.device, "hidden size:", encoder.config.hidden_size)

device: mps:0 hidden size: 1024


In [14]:
@torch.inference_mode()
def encode_pairs(prompts, responses, max_length=256):
    texts = [f"[PROMPT]\n{p}\n\n[RESPONSE]\n{r}" for p, r in zip(prompts, responses)]
    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(encoder.device)
    outputs = encoder(**batch)
    hidden = outputs.last_hidden_state

    mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp_min(1.0)
    pooled = summed / counts
    return pooled.float().cpu()

Mean-pooling the non-padding hidden states gives one deterministic vector per prompt-response pair. It is not the only reasonable pooling choice, but it is easy to explain and does not require any additional trained component on top of the frozen encoder.

In [15]:
fit_sample = fit_df.sample(n=min(N_FIT, len(fit_df)), random_state=42).reset_index(drop=True)
val_sample = val_df.sample(n=min(N_VAL, len(val_df)), random_state=42).reset_index(drop=True)
print("fit_sample:", fit_sample.shape, "val_sample:", val_sample.shape)

fit_sample: (500, 20) val_sample: (150, 20)


In [16]:
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

CACHE_DIR = Path("data/processed/reward_embeddings")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def encode_dataframe(df, batch_size=32):
    h0_parts, h1_parts = [], []
    for start in tqdm(range(0, len(df), batch_size), total=(len(df) + batch_size - 1) // batch_size):
        part = df.iloc[start:start + batch_size]
        h0_parts.append(encode_pairs(part["prompt"].tolist(), part["response_0"].tolist()).numpy())
        h1_parts.append(encode_pairs(part["prompt"].tolist(), part["response_1"].tolist()).numpy())
    return np.concatenate(h0_parts, axis=0), np.concatenate(h1_parts, axis=0)

h0_fit, h1_fit = encode_dataframe(fit_sample)
h0_val, h1_val = encode_dataframe(val_sample)

np.save(CACHE_DIR / "h0_fit.npy", h0_fit)
np.save(CACHE_DIR / "h1_fit.npy", h1_fit)
np.save(CACHE_DIR / "h0_val.npy", h0_val)
np.save(CACHE_DIR / "h1_val.npy", h1_val)
print(h0_fit.shape, h0_val.shape)

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

(500, 1024) (150, 1024)


Cache with provenance, not just the arrays: the model name/revision and the exact rows encoded belong alongside the `.npy` files, an embedding cache nobody can trace back to a model version is another dataset whose origin becomes unreconstructable later, exactly the lesson from Chapter 2 about label provenance, now applied to representations instead of labels.

### Turn response pairs into difference features

In [17]:
def make_difference_dataset(h0, h1, preference_ids):
    X = h0 - h1
    y = (np.asarray(preference_ids) == 0).astype(int)
    return X, y

X_help, y_help = make_difference_dataset(h0_fit, h1_fit, fit_sample["help_pref"])
print("response 0 helpfulness win rate:", (fit_sample["help_pref"] == 0).mean())

response 0 helpfulness win rate: 0.354


In [18]:
from sklearn.linear_model import LogisticRegression

# fit_intercept=False deliberately: if this represents one shared scalar reward r(h), the intercept
# cancels when we subtract two response rewards for the same head, so a learned intercept here
# could not actually be reproduced by the shared reward function we are claiming to build.
help_lr = LogisticRegression(C=1.0, fit_intercept=False, max_iter=1_000, random_state=42)
help_lr.fit(X_help, y_help)
w_help = help_lr.coef_[0]

In [19]:
def linear_reward(h, w):
    return np.asarray(h) @ np.asarray(w)

r0_val = linear_reward(h0_val, w_help)
r1_val = linear_reward(h1_val, w_help)
pred_response0 = (r0_val > r1_val).astype(int)

help_baseline_accuracy = (pred_response0 == (val_sample["help_pref"] == 0).to_numpy()).mean()
print("Helpfulness logistic-regression baseline, held-out accuracy:", help_baseline_accuracy)

Helpfulness logistic-regression baseline, held-out accuracy: 0.6


In [20]:
X_safe, y_safe = make_difference_dataset(h0_fit, h1_fit, fit_sample["safe_pref"])
safe_lr = LogisticRegression(C=1.0, fit_intercept=False, max_iter=1_000, random_state=42)
safe_lr.fit(X_safe, y_safe)
w_safe = safe_lr.coef_[0]

r0_safe_val = linear_reward(h0_val, w_safe)
r1_safe_val = linear_reward(h1_val, w_safe)
pred_response0_safe = (r0_safe_val > r1_safe_val).astype(int)
safe_baseline_accuracy = (pred_response0_safe == (val_sample["safe_pref"] == 0).to_numpy()).mean()
print("Safety logistic-regression baseline, held-out accuracy:", safe_baseline_accuracy)

Safety logistic-regression baseline, held-out accuracy: 0.5733333333333334


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

alignment = cosine_similarity(w_help.reshape(1, -1), w_safe.reshape(1, -1))[0, 0]
print("help/safety reward-direction cosine similarity:", alignment)

help/safety reward-direction cosine similarity: 0.46559262


A high positive cosine would suggest the frozen representation contains a broadly shared "good response" direction. A lower or negative value suggests real tension between the two learned directions, either way, this says something about the two directions *globally*, it does not tell us whether they conflict on the specific examples that matter most, that needs the conflict-row analysis in section 9.6.

## 9.5 Implement the reward head directly in PyTorch

Same idea, written the way modern post-training code usually writes it: a linear layer whose bias term cannot be identified by pairwise training (it cancels in every margin, exactly like the logistic-regression intercept above), trained with the actual Bradley-Terry loss rather than scikit-learn's logistic loss under the hood.

In [22]:
import torch.nn as nn

class LinearRewardHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)
    def forward(self, h):
        return self.score(h).squeeze(-1)

In [23]:
from torch.utils.data import Dataset, DataLoader

class PreferenceEmbeddingDataset(Dataset):
    def __init__(self, h0, h1, preference_ids):
        self.h0 = torch.tensor(h0, dtype=torch.float32)
        self.h1 = torch.tensor(h1, dtype=torch.float32)
        self.pref = torch.tensor(np.asarray(preference_ids), dtype=torch.long)
    def __len__(self):
        return len(self.pref)
    def __getitem__(self, idx):
        if self.pref[idx].item() == 0:
            return self.h0[idx], self.h1[idx]
        return self.h1[idx], self.h0[idx]

In [24]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, total_n = 0.0, 0
    for h_w, h_l in loader:
        optimizer.zero_grad()
        loss = pairwise_reward_loss(model(h_w), model(h_l))
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * len(h_w)
        total_n += len(h_w)
    return total_loss / total_n

@torch.inference_mode()
def evaluate_reward_head(model, h0, h1, preference_ids):
    model.eval()
    r0 = model(torch.tensor(h0, dtype=torch.float32)).numpy()
    r1 = model(torch.tensor(h1, dtype=torch.float32)).numpy()

    pref = np.asarray(preference_ids)
    true_response0 = pref == 0
    pred_response0 = r0 > r1
    accuracy = (pred_response0 == true_response0).mean()

    r_w = np.where(true_response0, r0, r1)
    r_l = np.where(true_response0, r1, r0)
    margins = r_w - r_l
    loss = np.logaddexp(0.0, -margins).mean()

    return {"accuracy": float(accuracy), "loss": float(loss), "r0": r0, "r1": r1, "margins": margins}

In [25]:
def train_reward_head(head, h0_fit, h1_fit, pref_fit, h0_val, h1_val, pref_val, epochs=20, lr=1e-3):
    loader = DataLoader(PreferenceEmbeddingDataset(h0_fit, h1_fit, pref_fit), batch_size=64, shuffle=True)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)

    best_state, best_val_loss = None, float("inf")
    for epoch in range(epochs):
        train_loss = train_one_epoch(head, loader, optimizer)
        val_result = evaluate_reward_head(head, h0_val, h1_val, pref_val)
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(epoch, f"train_loss={train_loss:.4f}", f"val_loss={val_result['loss']:.4f}", f"val_acc={val_result['accuracy']:.4f}")
        if val_result["loss"] < best_val_loss:
            best_val_loss = val_result["loss"]
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state)
    return head

hidden_size = h0_fit.shape[1]
help_head = LinearRewardHead(hidden_size)
help_head = train_reward_head(help_head, h0_fit, h1_fit, fit_sample["help_pref"], h0_val, h1_val, val_sample["help_pref"])

0 train_loss=0.7068 val_loss=0.7061 val_acc=0.5200
5 train_loss=0.5535 val_loss=0.6829 val_acc=0.6000
10 train_loss=0.4876 val_loss=0.6777 val_acc=0.5800
15 train_loss=0.4415 val_loss=0.6777 val_acc=0.6000
19 train_loss=0.4120 val_loss=0.6836 val_acc=0.6067


In [26]:
safe_head = LinearRewardHead(hidden_size)
safe_head = train_reward_head(safe_head, h0_fit, h1_fit, fit_sample["safe_pref"], h0_val, h1_val, val_sample["safe_pref"])

0 train_loss=0.7072 val_loss=0.7032 val_acc=0.5133


5 train_loss=0.5670 val_loss=0.6773 val_acc=0.5667


10 train_loss=0.5041 val_loss=0.6775 val_acc=0.5467
15 train_loss=0.4591 val_loss=0.6837 val_acc=0.5800
19 train_loss=0.4302 val_loss=0.6903 val_acc=0.5867


The PyTorch linear head should land close to the logistic-regression baseline's accuracy, they are the same model fit two different ways. A large discrepancy would be worth investigating (learning rate, standardisation, number of epochs) before adding any more complexity.

In [27]:
help_val_result = evaluate_reward_head(help_head, h0_val, h1_val, val_sample["help_pref"])
safe_val_result = evaluate_reward_head(safe_head, h0_val, h1_val, val_sample["safe_pref"])

print(f"Helpfulness: sklearn baseline={help_baseline_accuracy:.4f}, PyTorch head={help_val_result['accuracy']:.4f}")
print(f"Safety     : sklearn baseline={safe_baseline_accuracy:.4f}, PyTorch head={safe_val_result['accuracy']:.4f}")

Helpfulness: sklearn baseline=0.6000, PyTorch head=0.5867
Safety     : sklearn baseline=0.5733, PyTorch head=0.5533


### A small nonlinear head, as an extension rather than the default

In [28]:
class MLPRewardHead(nn.Module):
    def __init__(self, hidden_size, width=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden_size, width), nn.GELU(), nn.Dropout(dropout), nn.Linear(width, 1))
    def forward(self, h):
        return self.net(h).squeeze(-1)

help_mlp = MLPRewardHead(hidden_size)
help_mlp = train_reward_head(help_mlp, h0_fit, h1_fit, fit_sample["help_pref"], h0_val, h1_val, val_sample["help_pref"])
help_mlp_result = evaluate_reward_head(help_mlp, h0_val, h1_val, val_sample["help_pref"])
print(f"Helpfulness MLP head, held-out accuracy: {help_mlp_result['accuracy']:.4f} (linear head: {help_val_result['accuracy']:.4f})")

0 train_loss=0.6679 val_loss=0.6605 val_acc=0.6067


5 train_loss=0.4135 val_loss=0.7096 val_acc=0.6000


10 train_loss=0.2079 val_loss=0.8312 val_acc=0.5933


15 train_loss=0.0913 val_loss=0.9816 val_acc=0.5933


19 train_loss=0.0502 val_loss=1.0437 val_acc=0.5867
Helpfulness MLP head, held-out accuracy: 0.6067 (linear head: 0.5867)


A richer head can fit dataset-specific quirks as easily as genuine structure. If the MLP wins here but loses under the source-shift check in section 9.6, the extra flexibility bought overfitting, not generalisation, which is exactly why that comparison has to happen on the same prompt-disjoint validation set before either head is trusted.

## 9.6 Audit the reward model instead of printing one accuracy

### Margins tell us how the model is wrong, not just whether

In [29]:
margins = help_val_result["margins"]
print(np.quantile(margins, [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))

[-2.20767946 -1.24721993 -0.27208292  0.1981442   0.81693786  2.12677852
  2.81149289]


In [30]:
review_df = val_sample.copy()
review_df["margin_help"] = margins
review_df["correct_help"] = margins > 0

confident_errors = review_df[~review_df["correct_help"]].sort_values("margin_help").head(10)
confident_errors[["margin_help", "response_0_source", "response_1_source"]]

,margin_help,response_0_source,response_1_source
70,-4.006110,Alpaca-7B,Alpaca-7B
30,-2.336681,Alpaca3-8B,Alpaca3-8B
6,-2.073413,Alpaca2-7B,Alpaca2-7B
59,-1.408719,Alpaca-7B,Alpaca-7B
141,-1.378795,Alpaca2-7B,Alpaca2-7B
16,-1.315195,Alpaca3-8B,Alpaca3-8B
99,-1.279853,Alpaca-7B,Alpaca-7B
138,-1.274697,Alpaca-7B,Alpaca-7B
40,-1.213637,Alpaca2-7B,Alpaca2-7B
108,-1.166466,Alpaca-7B,Alpaca-7B


A wrong pair with margin -0.05 is nearly a tie. A wrong pair with margin -4 says the model confidently prefers the response the annotators rejected, that second kind of error is where a qualitative read is most likely to reveal a source-style shortcut, a truncated response, or genuine label ambiguity.

### Calibrate the pairwise probability, reusing Chapter 4's machinery

In [31]:
from scipy.special import expit
from sklearn.metrics import brier_score_loss, log_loss

# Symmetric construction: every pair contributes one (margin, target=1) and one (-margin, target=0)
# observation, so the reliability diagram is not trivially perfect just because every raw target is 1.
pair_margin = np.concatenate([margins, -margins])
pair_target = np.concatenate([np.ones(len(margins), dtype=int), np.zeros(len(margins), dtype=int)])
pair_prob = expit(pair_margin)

print("Brier:", brier_score_loss(pair_target, pair_prob))
print("Log loss:", log_loss(pair_target, pair_prob))

Brier: 0.23517324030399323
Log loss: 0.6747288703918457


In [32]:
from sklearn.calibration import calibration_curve

frac_correct, mean_conf = calibration_curve(pair_target, pair_prob, n_bins=10, strategy="quantile")
print("mean predicted prob vs observed frequency, by bin:")
for m, f in zip(mean_conf, frac_correct):
    print(f"  {m:.3f}  ->  {f:.3f}")

mean predicted prob vs observed frequency, by bin:
  0.145  ->  0.233
  0.275  ->  0.367
  0.363  ->  0.333
  0.424  ->  0.567
  0.472  ->  0.567
  0.528  ->  0.433
  0.576  ->  0.433
  0.637  ->  0.667
  0.725  ->  0.633
  0.855  ->  0.767


A reward model can rank correctly most of the time while being overconfident about *how much* it prefers the winner, exactly the discrimination-vs-calibration distinction from Chapters 4 and 7. That distinction starts to matter the moment anything downstream treats a reward difference as a strength-of-evidence signal rather than a bare ordering.

### Bootstrap the pairwise result

In [33]:
rng = np.random.default_rng(42)

def bootstrap_pairwise_accuracy(margins, n_boot=2_000):
    margins = np.asarray(margins)
    n = len(margins)
    values = [(margins[rng.integers(0, n, size=n)] > 0).mean() for _ in range(n_boot)]
    return np.quantile(values, [0.025, 0.50, 0.975])

print("Helpfulness accuracy 95% CI:", bootstrap_pairwise_accuracy(help_val_result["margins"]))
print("Safety accuracy 95% CI     :", bootstrap_pairwise_accuracy(safe_val_result["margins"]))

Helpfulness accuracy 95% CI: [0.50666667 0.58666667 0.66666667]
Safety accuracy 95% CI     : [0.47983333 0.55333333 0.63333333]


### Slice by safety structure, and by the exact rows where the two objectives disagree

In [34]:
def safety_pair_label(row):
    a, b = bool(row["is_response_0_safe"]), bool(row["is_response_1_safe"])
    if a and b: return "both_safe"
    if a and not b: return "0_safe_1_unsafe"
    if not a and b: return "0_unsafe_1_safe"
    return "both_unsafe"

review_df["safety_pair"] = review_df.apply(safety_pair_label, axis=1)
review_df["margin_safe"] = safe_val_result["margins"]
review_df["correct_safe"] = review_df["margin_safe"] > 0

safety_slice_report = review_df.groupby("safety_pair").agg(
    n=("correct_safe", "size"), safe_accuracy=("correct_safe", "mean"), mean_margin=("margin_safe", "mean"),
)
safety_slice_report

,n,safe_accuracy,mean_margin
safety_pair,,,
0_safe_1_unsafe,9,0.777778,1.031976
0_unsafe_1_safe,11,0.545455,0.150411
both_safe,61,0.573770,0.179425
both_unsafe,69,0.507246,0.031862


If the safety head is excellent whenever exactly one response is unsafe but weak when both share the same safety label, that is a specific, actionable finding, the overall accuracy number would have hidden it completely.

In [35]:
conflict_mask = ~val_sample["help_safe_agree"].to_numpy()
print(f"Conflict rows in validation: {conflict_mask.sum()} of {len(val_sample)}")
print("Helpfulness accuracy on conflict rows:", review_df.loc[conflict_mask, "correct_help"].mean())
print("Safety accuracy on conflict rows     :", review_df.loc[conflict_mask, "correct_safe"].mean())

Conflict rows in validation: 38 of 150
Helpfulness accuracy on conflict rows: 0.5789473684210527
Safety accuracy on conflict rows     : 0.5


A helpfulness head correctly predicting the helpfulness label on a conflict row, while actively preferring the response the safety head rejects, is not a bug, it is the two objectives encoding genuinely different preferences, exactly as intended. The failure to watch for is silently averaging these rows away once the two heads get combined into one policy signal in Chapter 10.

### Severity and response-source effects

In [36]:
review_df["max_severity"] = review_df[["response_0_severity_level", "response_1_severity_level"]].max(axis=1)

severity_report = review_df.groupby("max_severity").agg(n=("correct_safe", "size"), safe_accuracy=("correct_safe", "mean"))
severity_report

,n,safe_accuracy
max_severity,,
0,61,0.573770
1,6,0.500000
2,64,0.578125
3,19,0.421053


In [37]:
source_accuracy = (
    review_df.groupby(["response_0_source", "response_1_source"])
    .agg(n=("correct_help", "size"), help_accuracy=("correct_help", "mean"))
    .query("n >= 3")
    .sort_values("n", ascending=False)
)
source_accuracy.head(10)

,,n,help_accuracy
response_0_source,response_1_source,,
Alpaca2-7B,Alpaca2-7B,65,0.630769
Alpaca-7B,Alpaca-7B,55,0.600000
Alpaca3-8B,Alpaca3-8B,30,0.466667


A reward model whose accuracy is strong within a familiar source pairing but collapses for an unfamiliar one has likely learned some amount of generator style rather than a fully robust preference function, the same distribution-shift diagnosis from Chapter 5, now applied to preference data instead of a safety classifier.

## 9.7 Helpfulness and safety form a multi-objective problem

Scores from two independently trained heads are not automatically on comparable scales, combining raw values with a fixed weight can let whichever head happens to have the larger numerical range dominate regardless of the intended trade-off. Standardise each head's scores against a fixed validation reference before combining them, and remember that a chosen weight is a policy decision, not something the data can supply for you.

In [38]:
help_scores_val = help_val_result["r0"]
safe_scores_val = safe_val_result["r0"]

help_z = (help_scores_val - help_scores_val.mean()) / help_scores_val.std()
safe_z = (safe_scores_val - safe_scores_val.mean()) / safe_scores_val.std()

print("raw score ranges  :", help_scores_val.min(), help_scores_val.max(), "|", safe_scores_val.min(), safe_scores_val.max())
print("standardised ranges:", help_z.min(), help_z.max(), "|", safe_z.min(), safe_z.max())

raw score ranges  : -6.758777 3.523495 | -5.882532 0.6608658
standardised ranges: -2.7471297 2.621692 | -2.6657538 2.304116


In [39]:
def pareto_mask(help_scores, safe_scores):
    help_scores, safe_scores = np.asarray(help_scores), np.asarray(safe_scores)
    keep = np.ones(len(help_scores), dtype=bool)
    for i in range(len(help_scores)):
        dominated = (
            (help_scores >= help_scores[i]) & (safe_scores >= safe_scores[i])
            & ((help_scores > help_scores[i]) | (safe_scores > safe_scores[i]))
        )
        if dominated.any():
            keep[i] = False
    return keep

N_PARETO_SAMPLE = min(60, len(help_z))
frontier = pareto_mask(help_z[:N_PARETO_SAMPLE], safe_z[:N_PARETO_SAMPLE])
print(f"{frontier.sum()} of {N_PARETO_SAMPLE} sampled responses are on the Pareto frontier (not dominated on both axes at once).")

3 of 60 sampled responses are on the Pareto frontier (not dominated on both axes at once).


Every response on that frontier is one where no other sampled response is at least as good on both axes and strictly better on one, that is a materially more honest picture of the trade-off than a single scalarised score, which quietly bakes in one particular weighting the moment it is computed. An alternative worth knowing about: treat safety as a *constraint* (optimise helpfulness only among candidates whose safety score clears a minimum bar) rather than another weighted term, this mirrors the constrained-threshold design from Chapter 4, and it has the same appeal, instead of asking how many units of safety equal one unit of helpfulness, it asks for a defensible minimum acceptable region and optimises usefulness inside it. Either approach still needs `w` or the safety threshold to be a defensible, explicit choice, no amount of modelling makes that policy decision disappear on its own.

## 9.8 External generalisation: RewardBench 2

PKU-SafeRLHF taught the reward heads one dataset's annotation process and a limited set of response sources. RewardBench 2 asks whether that signal transfers: 1,865 prompts across Factuality, Precise Instruction Following, Math, Safety, Focus and Ties, each with one or more chosen responses to rank above several rejected ones.

In [40]:
rb2 = load_dataset("allenai/reward-bench-2", split="test")
print(rb2)
print(pd.Series(rb2["subset"]).value_counts())

Dataset({
    features: ['id', 'prompt', 'chosen', 'rejected', 'num_correct', 'num_incorrect', 'total_completions', 'models', 'subset', 'additional_metadata'],
    num_rows: 1865
})
Focus         495
Factuality    475
Safety        450
Math          183
Precise IF    160
Ties          102
Name: count, dtype: int64


In [41]:
# Ties needs the benchmark's own scoring rule (several valid answers, not a single chosen response),
# which is out of scope for this notebook, we evaluate the other five subsets, where success is
# defined as: every chosen response outranks every rejected response.
N_RB2 = 24
rb2_df = rb2.to_pandas()
rb2_sample = (
    rb2_df[rb2_df["subset"] != "Ties"]
    .groupby("subset", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), max(1, N_RB2 // 5)), random_state=42))
    .reset_index(drop=True)
)
print(rb2_sample["subset"].value_counts())

subset
Factuality    4
Focus         4
Math          4
Precise IF    4
Safety        4
Name: count, dtype: int64


In [42]:
def score_text_pairs(prompts, responses, reward_head):
    h = encode_pairs(prompts, responses)
    with torch.inference_mode():
        return reward_head(h).numpy()

def evaluate_rb2_example(example, reward_head):
    chosen, rejected = list(example["chosen"]), list(example["rejected"])
    chosen_scores = score_text_pairs([example["prompt"]] * len(chosen), chosen, reward_head)
    rejected_scores = score_text_pairs([example["prompt"]] * len(rejected), rejected, reward_head)
    success = chosen_scores.min() > rejected_scores.max()
    return {"success": bool(success), "margin": float(chosen_scores.min() - rejected_scores.max())}

In [43]:
rb2_results = []
for _, row in tqdm(rb2_sample.iterrows(), total=len(rb2_sample)):
    for head_name, head in [("helpfulness", help_head), ("safety", safe_head)]:
        result = evaluate_rb2_example(row, head)
        rb2_results.append({"subset": row["subset"], "reward_head": head_name, **result})

rb2_results_df = pd.DataFrame(rb2_results)
rb2_summary = rb2_results_df.groupby(["reward_head", "subset"])["success"].agg(["mean", "count"])
rb2_summary

  0%|          | 0/20 [00:00<?, ?it/s]

mean  count
reward_head subset                 
helpfulness Factuality  0.25      4
            Focus       0.25      4
            Math        0.00      4
            Precise IF  0.50      4
            Safety      0.25      4
safety      Factuality  0.25      4
            Focus       0.25      4
            Math        0.25      4
            Precise IF  0.50      4
            Safety      0.75      4

A harmlessness reward model transferring reasonably to the Safety subset while doing poorly on Math or Factuality would not be surprising, it would tell us, correctly, that this reward captures a narrower preference concept than a general-purpose assistant reward. Passive RewardBench accuracy is still not the regime we ultimately care about: it scores candidates drawn from ordinary generation, while Chapter 10 will use this exact reward signal to *optimise* a policy, pushing the response distribution towards whatever the reward model scores highest. If the reward contains a small exploitable correlation (an overvalued verbosity, confidence, refusal phrase, or source-model style), that is precisely the region optimisation pressure will find and amplify, and it is exactly the region today's passive dataset has the least evidence about.

### Practical exercise: build and audit two reward models

Using the pieces above, extend this into the chapter's full project:

1. Raise `N_FIT`/`N_VAL` toward the book's suggested 5,000-10,000 pairs and redo the bootstrap intervals, safety-combination slices, and severity/harm-category audit with real statistical power.
2. Add a genuine cross-source generalisation test: fit on one dominant response-source pairing, evaluate on a different one, and compare against the in-distribution accuracy from section 9.6.
3. Extend the harm-category audit past severity: expand `response_0_harm_category`/`response_1_harm_category` into a multilabel structure (as in Chapter 5) and report safety-head accuracy per category, remembering categories overlap.
4. Write down one concrete hypothesis about what an optimiser could exploit in this reward model (verbosity, hedging language, refusal phrasing, a specific source's style), based on the confident-error review in section 9.6, this is the hypothesis Chapter 10 will actually get to test.

Then write two short closing sections: **what the experiment supports** (a preference ordering that generalises to held-out PKU-SafeRLHF prompts with a measured, bootstrapped accuracy and known weaknesses) and **what it does not support** (that this reward model has learned human values, resolved the helpfulness/safety trade-off, or would transfer to a materially different deployment).

## Where we've arrived

We treated a reward model the way Chapters 3-4 treated a classifier: start simple, understand it completely, then add complexity. The simplest version, a linear reward on frozen representations, turned out to be logistic regression on differenced features, the same supervised-learning foundation as every earlier chapter, just with a relational target instead of an absolute one. We audited PKU-SafeRLHF before training anything (agreement rate, safety combinations, severity, source effects, a prompt-disjoint split), trained the same idea twice (scikit-learn and PyTorch) to confirm they agree, and then audited the result properly: margins and confident errors, calibration, bootstrap intervals, safety-structure and severity slices, and, most importantly, the exact rows where helpfulness and safety disagree rather than letting a combined score erase that disagreement silently. A Pareto-frontier view made the two-objective trade-off visible instead of collapsing it into one arbitrary weight, and RewardBench 2 gave us one external check on whether any of this generalises past PKU-SafeRLHF's own annotation process.

The reward model built here is not merely another supervised predictor, it is about to become the thing another optimiser actively tries to maximise. **Chapter 10** puts pressure on exactly that: we will optimise a policy against this reward signal and ask whether the small, predictable weaknesses uncovered in this chapter's audit turn into incentives once optimisation starts searching for them.